# OTTER on Google Colab

Run the real OTTER pipeline end to end in a free Colab runtime:

1. **Install** — download the published `otter-install` binary and run it for real, including
   the `enva` environment creation that installs `otter-core` and `otter-snakemake`.
2. **Simulate a reference genome** — build a reference *release* with the real registry
   layout, manifest, and checksums. The payloads are placeholders; nothing is downloaded.
3. **Author projects from real fixtures** — use the checked-in downsampled FASTQ from the
   repository, not synthetic data, and drive each scenario through `otter build`.
4. **Resolve the run** — execute `otter run --dry-run` for **every phase** of each workflow,
   so the executor compiles each phase's task graph and reports it, without running a single
   bioinformatics tool.

> **What this notebook does not do.** It never aligns a read or executes a workflow task.
> The reference genome is a *fixture*: real directory layout, real manifests, real digests,
> placeholder sequence. The `--dry-run` stops at Craftmake's planner. A dry run proves the
> run resolves and plans; it says nothing about scientific output.

Runtime: **CPU only**. A GPU gains nothing here.

## 0. Preflight

Confirm the runtime is Linux/x86-64 and that the tooling this notebook needs is present.
Colab already ships `git`, `python3`, and `curl`; the OTTER release binaries are static, so
nothing has to be compiled except one small helper the rehearsal needs.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
import time


def sh(command, check=True, capture=False, stream=False, echo=None, timeout=None):
    """Run a shell command, echoing it so the notebook reads as a transcript.

    capture=True collects output and prints it when the command finishes, which suits a
    short command whose output is a block to read. stream=True echoes output live, which
    is what a multi-minute step needs: a captured long command shows nothing and reads
    as a hang. echo=False captures without printing, for a command whose output the
    caller parses and summarises instead — a Craftmake plan envelope is hundreds of
    kilobytes of JSON, and printing it buries everything around it.
    """
    if echo is not None:
        capture = capture or not stream
    print(f"$ {command}")
    if stream:
        started = time.monotonic()
        process = subprocess.Popen(
            command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        for line in process.stdout:
            print(line.rstrip())
        returncode = process.wait(timeout=timeout)
        print(f"  (exit {returncode} after {time.monotonic() - started:.0f}s)")
        if check and returncode != 0:
            raise subprocess.CalledProcessError(returncode, command)
        return subprocess.CompletedProcess(command, returncode)

    result = subprocess.run(
        command,
        shell=True,
        check=check,
        text=True,
        capture_output=capture or echo is False,
        timeout=timeout,
    )
    if capture and echo is not False and result.stdout.strip():
        print(result.stdout.rstrip())
    return result


print("python :", sys.version.split()[0])
print("kernel :", platform.system(), platform.release(), platform.machine())
for tool in ("git", "curl", "tar"):
    print(f"{tool:7}:", shutil.which(tool) or "MISSING")

assert platform.system() == "Linux", "The OTTER release binaries are Linux builds."
assert platform.machine() in ("x86_64", "amd64"), (
    f"This notebook uses the amd64 release assets, not {platform.machine()}."
)
print("\nPreflight OK.")

### Configuration

Everything configurable lives in one place. The defaults are chosen so that the whole
notebook runs unattended in roughly fifteen minutes.

In [ ]:
from pathlib import Path

# The public repository that hosts the releases and the test fixtures.
REPO = "otterlab-bio/otter"
RELEASE_TAG = "latest"          # pin, e.g. "v1.1.0", for a reproducible run

# Where everything is staged. /content is Colab's persistent-for-the-session volume;
# /tmp is RAM-backed and would be lost between cells.
WORK = Path("/content/otter-colab")
INSTALL_DIR = WORK / "bin"       # holds the released binaries
REPO_DIR = WORK / "repo"         # a shallow clone, for fixtures and the e2e script
REGISTRY = WORK / "registry"     # the simulated reference registry
PROJECTS = WORK / "projects"     # one authored project per scenario

for directory in (WORK, INSTALL_DIR, REGISTRY, PROJECTS):
    directory.mkdir(parents=True, exist_ok=True)

print("repository :", REPO)
print("release    :", RELEASE_TAG)
print("work dir   :", WORK)

# The fixtures the scenarios are authored from. Each is a real downsampled library
# checked into the repository, paired with the reference role that satisfies it.
SCENARIOS = {
    "rrbs": {
        "mode": "RRBS",
        "accession": "SRR31480456",
        "references": {"primary": "hg19@GRCh37.p13-gencode-v19"},
    },
    "rnaseq": {
        "mode": "RNASEQ",
        "accession": "SRR018258",
        "references": {"primary": "hg38@GRCh38.p14"},
    },
    "bs-pdx": {
        "mode": "RRBS",
        "accession": "SRR36187610",
        "references": {
            "graft": "hg38@GRCh38.p14",
            "host": "mm10@GRCm38.p6",
        },
    },
    "rna-pdx": {
        "mode": "RNASEQ",
        "accession": "SRR30880970",
        "references": {
            "graft": "hg38@GRCh38.p14",
            "host": "mm10@GRCm38.p6",
        },
    },
}

print("\nscenarios:", ", ".join(SCENARIOS))

## 1. Install with the real `otter-install`

This downloads the published installer and runs it. It is not a simulation: it resolves the
release, fetches every static binary, deploys the Craftmake workflow catalog, and creates the
`enva` environments.

**Environment creation is the slow part** — plan on a few minutes for `otter-core`. That is
`conda` solving and downloading a real bioinformatics stack, not overhead in this notebook.

Two flags are worth knowing:

- `-install-dir` keeps the whole install inside the Colab session instead of a home directory.
- `-non-interactive` takes every default, which is what makes this cell unattended.

In [ ]:
installer_url = (
    f"https://github.com/{REPO}/releases/{RELEASE_TAG}/download/"
    "otter-install-linux-amd64-static"
)
installer_path = WORK / "otter-install"

sh(f"curl -fsSL -o {installer_path} {installer_url}")
installer_path.chmod(0o755)
print(f"\ninstaller: {installer_path} ({installer_path.stat().st_size:,} bytes)")

### Inspect the plan before changing anything

`-dry-run` prints every action it would take. Reading it first is how you tell an installer
problem from a network problem later.

In [ ]:
# Assign to a name rather than leaving the call as the cell's last expression. A bare
# call in a Jupyter cell is auto-displayed as its repr, which re-prints the whole
# captured stdout as one escaped line and buries the readable output above it.
plan = sh(
    f"{installer_path} -dry-run -non-interactive "
    f"-releases-repo {REPO} -install-dir {INSTALL_DIR}",
    capture=True,
)

### Run the install

`-skip-envs` is **not** passed, so environments are created. If you want a fast pass that only
fetches the binaries, set `SKIP_ENVS = True` below — the rest of the notebook still works,
because the authoring and planning steps do not need a tool environment to be present.

In [ ]:
SKIP_ENVS = False  # set True to skip the multi-minute environment creation

skip_flag = "-skip-envs" if SKIP_ENVS else ""

# stream=True, not capture=True: this step downloads a full bioinformatics stack and
# can take several minutes. Capturing it would show nothing until it finished and read
# as a hang. Assigned rather than left bare so Jupyter does not also display its repr.
install = sh(
    f"{installer_path} -non-interactive {skip_flag} "
    f"-releases-repo {REPO} -install-dir {INSTALL_DIR}",
    stream=True,
)

### Verify what landed

Print the installed tools and the environments, so a partial install is visible here rather
than surfacing as a confusing failure three cells later.

In [ ]:
os.environ["PATH"] = f"{INSTALL_DIR}:{os.environ['PATH']}"

print("=== installed binaries ===")
for binary in sorted(INSTALL_DIR.iterdir()):
    if binary.is_file() and os.access(binary, os.X_OK):
        print(f"  {binary.name}")

print("\n=== otter and craftmake versions ===")
sh("otter --version", capture=True)
sh("craftmake --version", capture=True)

print("=== workflow catalog deployed by the installer ===")
catalog = INSTALL_DIR / "workflows"
print(f"  {catalog}: {sorted(p.name for p in catalog.iterdir()) if catalog.is_dir() else 'MISSING'}")

In [ ]:
print("=== enva environments ===")
if SKIP_ENVS:
    print("  skipped (-skip-envs)")
else:
    sh("enva list", capture=True, check=False)

    # Prove otter-core is usable rather than merely listed: this is the difference
    # between "an environment directory exists" and "the environment works".
    print("\n=== does otter-core actually run a tool? ===")
    result = sh("enva run otter-core -- fastqc --version", capture=True, check=False)
    print("  reachable:", result.returncode == 0)

## 2. Fetch the fixtures and the rehearsal helper

Two things have to come from the repository rather than the release:

- the **downsampled FASTQ fixtures** under `testdata/`, which the scenarios are authored from;
- **`stub-registry`**, a small Go program that writes a reference release using the production
  layout and then verifies it with the production verifier.

Only that one helper needs compiling, so this cell installs Go if it is missing. The shallow
clone and the single Go build together take well under a minute.

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    # Shallow and without submodules: this clone is only for the fixtures and the
    # rehearsal script, and the toolchain comes from the release.
    sh(
        f"git clone --depth 1 --no-recurse-submodules "
        f"https://github.com/{REPO}.git {REPO_DIR}",
        stream=True,
    )
else:
    print(f"reusing clone at {REPO_DIR}")

print()
sh(f"git -C {REPO_DIR} log --oneline -1", capture=True)

fixture_root = REPO_DIR / "testdata/gate6/craftmake-downsample-20260906/fastq"
print(f"\nfixtures: {fixture_root}")
print("  accessions:", sorted(p.name for p in fixture_root.iterdir()))

In [ ]:
# stub-registry is the only program this notebook compiles.
if shutil.which("go") is None:
    print("Go not found; installing it (a few seconds).")
    sh("apt-get -qq update && apt-get -qq install -y golang-go", check=False)

go_path = shutil.which("go")
print("go:", go_path or "STILL MISSING")
if go_path:
    # Assigned rather than left bare so Jupyter does not also display its repr.
    go_version = sh("go version", capture=True)

In [ ]:
stub_registry = INSTALL_DIR / "stub-registry"

if go_path is None:
    raise RuntimeError(
        "stub-registry needs a Go toolchain. Install Go, or pre-build the binary and place "
        f"it at {stub_registry}."
    )

sh(f"cd {REPO_DIR} && go build -o {stub_registry} ./internal/e2esupport/cmd/stub-registry")
print(f"\nstub-registry: {stub_registry.stat().st_size:,} bytes")

## 3. Simulate the reference genome

A canonical project does not point at a FASTA path. It locks a **registry release**: a logical
id, a release label, and the manifest digest of everything in that release. So the fastest way
to satisfy a project is to write a release.

`stub-registry` does exactly this, and then runs the **production** reference verifier over
its own output. What you get is a real registry layout — `reference.yaml`, `manifest.json`,
`checksums.sha256`, `fasta/`, `annotations/`, `indexes/` — whose payloads are placeholders.

That distinction is the whole point: **the identity machinery is real, the sequence is not.**
Any claim this notebook makes about a run is therefore about resolution and planning, never
about biology.

In [ ]:
# Every reference the scenarios select. Both species are needed because the PDX scenarios
# declare a graft and a host.
REFERENCES = [
    {
        "id": "hg19",
        "release": "GRCh37.p13-gencode-v19",
        "organism": "Homo sapiens",
        "assembly": "GRCh37.p13",
        "aliases": "hg19,human,grch37",
    },
    {
        "id": "hg38",
        "release": "GRCh38.p14",
        "organism": "Homo sapiens",
        "assembly": "GRCh38",
        "aliases": "hg38,human,grch38",
    },
    {
        "id": "mm10",
        "release": "GRCm38.p6",
        "organism": "Mus musculus",
        "assembly": "GRCm38",
        "aliases": "mm10,mouse,grcm38",
    },
]

for reference in REFERENCES:
    sh(
        f"{stub_registry} --registry-root {REGISTRY} "
        f"--id {reference['id']} --release {reference['release']} "
        f"--organism '{reference['organism']}' --assembly {reference['assembly']} "
        f"--alias {reference['aliases']}"
    )
    print()

In [ ]:
import json

print("=== registry layout ===")
sh(f"find {REGISTRY} -maxdepth 4 -mindepth 3 | sort", capture=True)

release_dir = REGISTRY / "genomes/hg19/GRCh37.p13-gencode-v19"
print("=== reference.yaml (the release identity) ===")
print("\n".join(
    f"  {line}" for line in (release_dir / "reference.yaml").read_text().splitlines()[:14]
))

manifest = json.loads((release_dir / "manifest.json").read_text())
print(f"\n=== manifest.json: {len(manifest)} entries ===")
for entry in manifest[:5]:
    print(f"  {entry['path']}")
print(f"  ... {max(0, len(manifest) - 5)} more")

# The release contract is reference.yaml + manifest.json + checksums.sha256. Report
# which of them the fixture wrote rather than assuming all three: the stub and the
# production publisher are separate code paths, and a notebook that asserts one
# implementation's output goes down on the other's.
print("\n=== contract files written by the fixture ===")
contract_files = ["reference.yaml", "manifest.json", "checksums.sha256"]
for name in contract_files:
    path = release_dir / name
    if path.is_file():
        print(f"  {name:<20} {path.stat().st_size:>6} bytes")
    else:
        print(f"  {name:<20} MISSING")

checksums_path = release_dir / "checksums.sha256"
if checksums_path.is_file():
    print("\n=== checksums.sha256 covers the release identity ===")
    print("\n".join(
        f"  {line}"
        for line in checksums_path.read_text().splitlines()[:4]
    ))
    print("\n  Verified against every file it names:")
    # Assigned, not left bare: a bare call as the cell's last expression is
    # auto-displayed as its repr, which re-prints stdout as one escaped line.
    checksum_check = sh(f"cd {release_dir} && sha256sum -c checksums.sha256 | tail -3", capture=True)
else:
    print(
        "\nchecksums.sha256 is absent, so sha256sum -c cannot be demonstrated here."
        "\nThe file is part of the release contract, so its absence is a fixture gap:"
        "\nreport it rather than treating the release as complete."
    )

## 4. Author every scenario with `otter build`

For each scenario this cell:

1. stages the real fixture under the `*_R1.fastq.gz` / `*_R2.fastq.gz` naming `create` discovers,
   and writes the matching pdata file;
2. runs **`otter build`** — the single command that chains `init`, `create`, `config validate`,
   and `config resolve`, then stops before execution;
3. checks that the project, samples manifest, and reference lock were actually written.

`build` is deliberately a shortcut over the same functions the standalone commands call, not a
second authoring path. The repository's offline rehearsal asserts the two produce byte-identical
authoring artifacts.

In [ ]:
def stage_inputs(scenario, accession):
    """Copy one fixture pair under the names `otter create` expects, plus its pdata."""
    project_dir = PROJECTS / scenario
    fastq_dir = project_dir / "fastq"
    fastq_dir.mkdir(parents=True, exist_ok=True)

    source = fixture_root / accession
    shutil.copyfile(source / "R1.fastq.gz", fastq_dir / f"{accession}_R1.fastq.gz")
    shutil.copyfile(source / "R2.fastq.gz", fastq_dir / f"{accession}_R2.fastq.gz")

    pdata = project_dir / "pdata.csv"
    pdata.write_text(
        "sampleid,inline_barcode_sequence,condition\n" f"{accession},,case\n"
    )
    return project_dir, fastq_dir, pdata


def reference_arguments(references):
    """Render the role flags. A primary is one role; PDX is graft plus host."""
    if "primary" in references:
        return f"--reference-primary {references['primary']}"
    return (
        f"--reference-graft {references['graft']} "
        f"--reference-host {references['host']}"
    )


snapshots = {}

for scenario, spec in SCENARIOS.items():
    print("=" * 72)
    print(f"scenario: {scenario}  (mode={spec['mode']}, fixture={spec['accession']})")
    print("=" * 72)

    project_dir, fastq_dir, pdata = stage_inputs(scenario, spec["accession"])
    references = reference_arguments(spec["references"])

    result = sh(
        f"otter build --project-root {project_dir} "
        f"--fastq {fastq_dir} --pdata {pdata} --mode {spec['mode']} "
        f"--jobid {scenario} "
        f"--reference-root {REGISTRY} {references} --backend local",
        capture=True,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(f"otter build failed for {scenario} (exit {result.returncode})")

    # `build` prints the snapshot path on its own line, last.
    snapshot_path = Path(result.stdout.strip().splitlines()[-1].strip())
    assert snapshot_path.name == "run.yaml", f"unexpected snapshot path: {snapshot_path}"
    assert snapshot_path.exists(), f"reported snapshot does not exist: {snapshot_path}"
    snapshots[scenario] = snapshot_path

    for artifact in ("project.yaml", "samples.tsv", "references.lock.yaml"):
        assert (project_dir / artifact).exists(), f"{scenario} is missing {artifact}"

    print(f"  snapshot: {snapshot_path.relative_to(WORK)}")
    print()

print(f"authored {len(snapshots)} scenarios: {', '.join(snapshots)}")

### What `build` produced

The project directory is worth reading once. Note where the Snakemake rules are pinned: under
`workflows/rules/`, beside the Snakefiles, because a Snakefile's `include:` directives resolve
relative to the Snakefile itself. Nothing is written into the project root.

In [ ]:
sample_project = PROJECTS / "rrbs"

print("=== project layout ===")
sh(f"find {sample_project} -maxdepth 2 -type d | sort", capture=True)

print("=== project.yaml ===")
print("\n".join(
    f"  {line}" for line in (sample_project / "project.yaml").read_text().splitlines()
))

print("\n=== samples.tsv (paths are project-relative) ===")
print("\n".join(
    f"  {line}" for line in (sample_project / "samples.tsv").read_text().splitlines()
))

print("\n=== references.lock.yaml (the locked digest) ===")
print("\n".join(
    f"  {line}"
    for line in (sample_project / "references.lock.yaml").read_text().splitlines()
))

print("\n=== the rules are pinned under workflows/, not at the root ===")
print("  workflows/rules exists:", (sample_project / "workflows/rules").is_dir())
print("  project-root rules/ exists:", (sample_project / "rules").exists())

## 5. Resolve every run with `--dry-run`

This is the step that answers *"does it actually resolve?"*.

`--dry-run` maps to `craftmake plan`: the executor validates the immutable snapshot, compiles
the DAG against the workflow catalog the installer deployed, and reports the task graph. It
touches no input data and runs no tool.

Three things are asserted for each scenario, so a passing cell means something specific:

- the process exits zero;
- it returns a real protocol envelope with `"command": "plan"` and `"ok": true`;
- the envelope is non-empty, so it planned tasks rather than merely succeeding.

In [ ]:
import json

# Named explicitly rather than left to PATH: the resolver falls back to a PATH
# lookup, and a notebook that reorders its environment should not fail on that.
craftmake_binary = INSTALL_DIR / "craftmake"
catalog = INSTALL_DIR / "workflows"


def phases_for(workflow):
    """List a workflow's published phases, in order, from the deployed catalog.

    Read from the catalog rather than hardcoded: the phase set differs per workflow
    (RNA-seq has no step3), and a hardcoded list would silently stop covering a new
    phase the moment one is added.
    """
    order = {"step1": 0, "step2": 1, "step2-check": 2, "step3": 3, "step3-check": 4, "publish": 5}
    names = [p.stem for p in (catalog / workflow).glob("*.yaml")]
    return sorted(names, key=lambda name: (order.get(name, 99), name))


def plan_phase(scenario, snapshot_path, phase):
    """Plan one phase and return its envelope.

    echo=False, because a plan envelope is hundreds of kilobytes of JSON: printing it
    would bury the summary this cell exists to produce. The task count is the useful
    part, and it is printed below.
    """
    result = sh(
        f"otter run --config {snapshot_path} "
        f"--executor craftmake --phase {phase} "
        f"--dry-run --foreground "
        f"--craftmake-binary {craftmake_binary} --catalog {catalog}",
        echo=False,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"dry run failed for {scenario} {phase} (exit {result.returncode})")
    return json.loads(result.stdout)


# Every phase is planned, not just step1. A run resolves to a multi-phase workflow,
# so planning one phase proves one phase; the later phases consume the earlier ones'
# outputs, and that is where a resolution problem actually shows up.
plan_results = {}
total_tasks = 0

for scenario, snapshot_path in snapshots.items():
    print("=" * 72)
    print(f"planning: {scenario}")
    print("=" * 72)

    # Discover the workflow from the snapshot's own step1 plan, so the phase list
    # comes from what this run actually selects rather than from a naming convention.
    first_envelope = plan_phase(scenario, snapshot_path, "step1")
    workflow = first_envelope["data"]["workflow"]
    phase_names = phases_for(workflow)
    print(f"  workflow: {workflow}")
    print(f"  phases  : {', '.join(phase_names)}")
    print()

    scenario_plans = {}
    for phase in phase_names:
        envelope = (
            first_envelope
            if phase == "step1"
            else plan_phase(scenario, snapshot_path, phase)
        )

        assert envelope.get("command") == "plan", envelope.get("command")
        assert envelope.get("ok") is True, envelope
        tasks = envelope.get("data", {}).get("tasks", [])
        assert tasks, f"{scenario} {phase} returned no tasks"

        scenario_plans[phase] = len(tasks)
        total_tasks += len(tasks)
        print(f"    {phase:<12} {len(tasks):>3} tasks")
    print()

    plan_results[scenario] = {"workflow": workflow, "phases": scenario_plans}

print(f"planned every phase of {len(plan_results)} scenarios, {total_tasks} tasks in total")
print("no task was executed")

## 6. Summary

A compact table of what was installed, authored, and planned.

In [ ]:
from datetime import datetime, timezone

print("=" * 72)
print("OTTER Colab walkthrough -- summary")
print("=" * 72)
print(f"finished     : {datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S} UTC")
print(f"repository   : {REPO}")
print(f"release      : {RELEASE_TAG}")
print(f"environments : {'skipped' if SKIP_ENVS else 'otter-core (created)'}")
print()

header = f"{'scenario':<10} {'mode':<8} {'fixture':<12} {'workflow':<17} {'phases':>6} {'tasks':>6}"
print(header)
print("-" * len(header))
for scenario, spec in SCENARIOS.items():
    result = plan_results[scenario]
    phase_tasks = result["phases"]
    print(
        f"{scenario:<10} {spec['mode']:<8} {spec['accession']:<12} "
        f"{result['workflow']:<17} {len(phase_tasks):>6} {sum(phase_tasks.values()):>6}"
    )

print()
print("Every scenario resolved to an immutable snapshot, and every phase of each")
print("workflow reached Craftmake's planner. No bioinformatics tool was executed and")
print("no result is scientific.")
print()
print(f"artifacts under {WORK}:")
print(f"  {INSTALL_DIR}   released binaries")
print(f"  {REGISTRY}      simulated reference registry")
print(f"  {PROJECTS}      authored projects and resolved runs")

## Where to go next

**Run the full offline rehearsal.** The same repository ships `scripts/e2e/otter_e2e.sh`, which
drives 103 stages across all four scenarios, the legacy migration path, the site-profile leg,
and the executor pairing contract — including a leg that authors every scenario twice and
compares `otter build` against the manual `init` → `create` → `resolve` chain artifact by
artifact.

```bash
cd /content/otter-colab/repo
bash scripts/e2e/otter_e2e.sh \
  --otter        /content/otter-colab/bin/otter \
  --craftmake    /content/otter-colab/bin/craftmake \
  --stub-registry /content/otter-colab/bin/stub-registry \
  --installer    /content/otter-colab/otter-install \
  --craftmake-catalog /content/otter-colab/bin/workflows
```

It is offline: it downloads no genome, builds no index, and executes no workflow task.

**Use a real reference genome.** Replace the simulated registry with a published release:

```bash
otter-install -reference-fetch \
  -releases-repo otterlab-bio/otter \
  -install-dir /content/otter-colab/bin
```

**Read the manual.** The [user manual](https://github.com/otterlab-bio/otter/blob/main/docs/manual/README.md)
covers installation, data preparation, every analysis mode, site profiles, and migration from
the legacy layout.

**Common snags.** If `otter` is not found, the `PATH` export in the verification cell was
skipped or the runtime restarted — re-run it. If environment creation failed, the rest of the
notebook still works: authoring and planning do not require a tool environment.